# Evaluation

Team 16 - Gunn Madan, Harini Mohan

1. Precision / recall / F1 of each detector vs. extreme-percentile proxy labels.
2. Pairwise Jaccard agreement between detectors.
3. Stability (anomaly rate per year) to check for drift.
4. Qualitative check against known severe weather events.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

import pandas as pd
import matplotlib.pyplot as plt

import evaluation as ev
import visualize as vz

DATA_DIR = os.path.abspath(os.path.join('..', 'data'))
FIG_DIR = os.path.abspath(os.path.join('..', 'outputs', 'figures'))
TABLE_DIR = os.path.abspath(os.path.join('..', 'outputs', 'tables'))

scored = pd.read_csv(os.path.join(DATA_DIR, 'model_output_data.csv'))
scored['DATE'] = pd.to_datetime(scored['DATE'])
full = pd.read_csv(os.path.join(DATA_DIR, 'model_output_full.csv'))
full['DATE'] = pd.to_datetime(full['DATE'])
scored.shape, full.shape

((3517, 38), (11170, 38))

## 1. Precision / recall / F1 against extreme-percentile proxy labels

Proxy labels: the top 2% (or top 1% + bottom 1%) of each `(CITY, variable)` in the test window are treated as 'true' extreme events. Precision reflects how often a flagged day is actually extreme; recall reflects how many real extremes the method catches.

In [2]:
metrics = ev.evaluate_methods(scored)
metrics.round(3)

,city,variable,method,flagged,tp,fp,fn,precision,recall,f1
0,Chicago,TMAX,Z-score,134,18,116,17,0.134,0.514,0.213
1,Chicago,TMAX,MAD,86,10,76,25,0.116,0.286,0.165
2,Chicago,TMAX,IsolationForest,47,13,34,22,0.277,0.371,0.317
3,Chicago,TMIN,Z-score,138,16,122,9,0.116,0.640,0.196
4,Chicago,TMIN,MAD,89,7,82,18,0.079,0.280,0.123
5,Chicago,TMIN,IsolationForest,47,12,35,13,0.255,0.480,0.333
6,Chicago,PRCP,Z-score,100,23,77,1,0.230,0.958,0.371
7,Chicago,PRCP,MAD,42,7,35,17,0.167,0.292,0.212
8,Chicago,PRCP,IsolationForest,47,7,40,17,0.149,0.292,0.197
9,LA,TMAX,Z-score,90,15,75,12,0.167,0.556,0.256


In [3]:
summary = (metrics.groupby(['variable', 'method'])
           [['precision', 'recall', 'f1']].mean().round(3))
summary

precision  recall     f1
variable method                                   
PRCP     IsolationForest      0.620   0.403  0.455
         MAD                  0.279   0.222  0.171
         Z-score              0.248   0.819  0.367
TMAX     IsolationForest      0.205   0.169  0.170
         MAD                  0.148   0.333  0.205
         Z-score              0.155   0.544  0.241
TMIN     IsolationForest      0.161   0.199  0.161
         MAD                  0.071   0.194  0.103
         Z-score              0.111   0.544  0.184

In [4]:
vz.plot_precision_bars(metrics, FIG_DIR)

'/sessions/dreamy-optimistic-cray/mnt/outputs/Team16-WeatherAnomalyDetection/outputs/figures/eval_precision_bars.png'

## 2. Detector agreement (Jaccard)

In [5]:
for var in ['TMAX', 'TMIN', 'PRCP']:
    mat = ev.agreement_matrix(scored, var)
    print(var)
    print(mat.round(3))
    vz.plot_agreement_heatmap(mat, var, FIG_DIR)

TMAX
                 Z-score    MAD  IsolationForest
Z-score            1.000  0.260            0.054
MAD                0.260  1.000            0.051
IsolationForest    0.054  0.051            1.000
TMIN
                 Z-score    MAD  IsolationForest
Z-score            1.000  0.273            0.030
MAD                0.273  1.000            0.029
IsolationForest    0.030  0.029            1.000
PRCP
                 Z-score    MAD  IsolationForest
Z-score            1.000  0.097            0.106
MAD                0.097  1.000            0.045
IsolationForest    0.106  0.045            1.000


## 3. Stability over time

Flagged fraction per year per city per method. A roughly flat line means the detector is stable; a jump around the train/test boundary is a useful signal that the underlying distribution has shifted.

In [6]:
for var in ['TMAX', 'TMIN', 'PRCP']:
    stab = ev.stability_over_time(full, var)
    vz.plot_stability(stab, var, FIG_DIR)
    print(var, 'saved')

TMAX saved
TMIN saved


PRCP saved


## 4. Top anomalies (qualitative check)

High-magnitude flagged days lined up against public records. Specific matches discussed in the final report:
- **2023-07-02 Chicago PRCP 3.35 in** - well-documented severe thunderstorm day.
- **2023-09-29 NYC PRCP 5.48 in** - historic Tri-State flood event.
- **2024-01-16 Chicago TMAX 3 F** - Midwest cold wave.
- **2025 May LA TMAX >90 F for multiple days** - SoCal early-season heat wave.

In [7]:
for var in ['TMAX', 'TMIN', 'PRCP']:
    print('Top', var)
    print(ev.top_anomalies(scored, var, method='Z-score', top_n=10).to_string(index=False))
    print()

Top TMAX
      DATE    CITY  TMAX    z_TMAX  abs_score
2025-05-30      LA  96.0 12.223148  12.223148
2025-05-31      LA  91.0  9.269627   9.269627
2024-01-16 Chicago   3.0 -6.939315   6.939315
2025-05-10      LA  99.0  6.258922   6.258922
2024-07-18 Chicago  75.0 -5.184868   5.184868
2025-06-24     NYC  99.0  5.154143   5.154143
2024-09-09      LA 105.0  5.151927   5.151927
2025-05-11      LA  90.0  5.143521   5.143521
2023-09-17 Chicago  64.0 -5.009998   5.009998
2023-01-16 Chicago  52.0  4.961610   4.961610

Top TMIN
      DATE    CITY  TMIN    z_TMIN  abs_score
2024-08-21     NYC  57.0 -7.977647   7.977647
2024-04-05      LA  47.0 -7.794229   7.794229
2025-08-22     NYC  58.0 -7.413571   7.413571
2023-02-16     NYC  56.0  7.343200   7.343200
2023-10-26 Chicago  65.0  7.043637   7.043637
2025-06-23 Chicago  79.0  6.939315   6.939315
2025-01-19      LA  42.0 -6.647835   6.647835
2025-12-05 Chicago   9.0 -6.617763   6.617763
2023-01-19      LA  43.0 -6.051234   6.051234
2024-04-06     

## 5. Per-city anomaly time series

Overlays the flags from each method on the raw signal so you can eyeball where detectors agree and where they diverge.

In [8]:
for var in ['TMAX', 'TMIN', 'PRCP']:
    for city in ['NYC', 'Chicago', 'LA']:
        for method in ['Z-score', 'MAD', 'IsolationForest']:
            vz.plot_anomaly_timeseries(full, city, var, method, FIG_DIR)
print('All anomaly time series saved to', FIG_DIR)

All anomaly time series saved to /sessions/dreamy-optimistic-cray/mnt/outputs/Team16-WeatherAnomalyDetection/outputs/figures
